# Oppgave 1: Geokoding (10 poeng)

Det overordnede målet med oppgavene a)-e) er å finne ut **hvor mange mennesker som bor innen gangavstand (1,5 km) fra visse kjøpesentre i Oslo**.

Oppgave 1 gjelder lokasjonene til kjøpesentrene: finn adressene til kjøpesenterne og konverter de til koordinater.

### a) Forbered en inputfil som inneholder adressene til kjøpesentre

Finn ut adressene til følgende kjøpesentre (f.eks. ved å bruke din favorittsøkemotor), og samle dem i en tekstfil kalt `shopping_centres.txt`:

 - Oslo City
 - Bryn Senter
 - Storo Storsenter
 - Lambertseter senter
 - Manglerud Senter
 - Linderud senter
 - Tveita Senter
 
Tekstfilen skal være i semikolon-separert format (`;`) og inkludere følgende kolonner:

- `id` (integer) en unik identifikator for hvert kjøpesenter
- `navn` (string) navnet på hvert kjøpesenter
- `adr` (string) adressen

Se et eksempel på hvordan du formaterer tekstfilen [fra forelesningen](https://haavardaagesen.github.io/gmgi221/content/notebooks/04_geokoding-i-geopandas.html).


### b) Les in listen med adresser

Les inn listen med adresser du nettopp forberedte inn i en `pandas.DataFrame` kalt `shopping_centres`

In [2]:
### BEGIN SOLUTION
import pathlib
DATA_DIRECTORY = pathlib.Path().resolve() / "data"

import pandas
shopping_centres = pandas.read_csv(DATA_DIRECTORY / "shopping_centres.txt", sep=";")

### END SOLUTION

In [3]:
# NON-EDITABLE CODE CELL FOR TESTING YOUR SOLUTION
### BEGIN HIDDEN TESTS
import pandas
assert isinstance(shopping_centres, pandas.DataFrame)
for column in ("id", "navn", "adr"):
    assert column in shopping_centres.columns
### END HIDDEN TESTS

### c) Geokode adressene

Du skal nå geokode adressene ved å bruke Nominatim sin geokodingstjenesten. Slå sammen resultatene med inputdataene, og lagre dem i en `geopandas.GeoDataFrame` med samme navn (`shopping_centres`).

Husk å definere en tilpasset `user_agent`-streng!

In [4]:
### BEGIN SOLUTION
import geopandas

shopping_centres_coordinates = geopandas.tools.geocode(
    shopping_centres["adr"],
    provider="nominatim",
    user_agent="gmgi221"
)

shopping_centres = shopping_centres_coordinates.join(shopping_centres)[["id", "navn", "adr", "geometry"]]
### END SOLUTION

In [5]:
# NON-EDITABLE CODE CELL FOR TESTING YOUR SOLUTION
### BEGIN HIDDEN TESTS
import geopandas
assert isinstance(shopping_centres, geopandas.GeoDataFrame)
for column in ("id", "navn", "adr", "geometry"):
    assert column in shopping_centres.columns
### END HIDDEN TESTS

Sjekk at koordinatsystemet til det geokodede resultatet er korrekt definert, og **reprojiser laget til ETRS89** (EPSG:25832):

In [6]:
### BEGIN SOLUTION
shopping_centres.crs = "EPSG:4326"
shopping_centres = shopping_centres.to_crs("EPSG:25832")
### END SOLUTION

In [7]:
# NON-EDITABLE CODE CELL FOR TESTING YOUR SOLUTION
### BEGIN HIDDEN TESTS
import pyproj
assert shopping_centres.crs == pyproj.CRS("EPSG:25832")
### END HIDDEN TESTS

### d) Opprett en *buffer* rundt punktene

Beregn en 1,5 km buffer for hvert geokodede punkt. Overskriv `geometry`-kolonnen med den nye buffergeometrien.

Bruk [`geopandas.GeoDataFrame.buffer()`-metoden](http://geopandas.org/geometric_manipulations.html#GeoSeries.buffer), som bruker shapely’s [`buffer()`](http://toblerity.org/shapely/manual.html#object.buffer) i bakgrunnen. Du trenger bare å bry deg om `distance`-parameteren, ikke bekymre deg for de andre mulige argumentene.

In [8]:
### BEGIN SOLUTION
shopping_centres["geometry"] = shopping_centres["geometry"].buffer(1500)
### END SOLUTION

In [9]:
# NON-EDITABLE CODE CELL FOR TESTING YOUR SOLUTION
### BEGIN HIDDEN TESTS
assert shopping_centres.geometry.geom_type.unique() == ["Polygon"]
### END HIDDEN TESTS

### e) Lagre buffergeometrilaget

Lagre dataframen som inneholder buffergeometriene i en *GeoJSON*-fil med navn `shopping_centres.geojson`:

In [10]:
### BEGIN SOLUTION
shopping_centres.to_file('shopping_centres.geojson')
### END SOLUTION

In [ ]:
# NON-EDITABLE CODE CELL FOR TESTING YOUR SOLUTION
### BEGIN HIDDEN TESTS

### END HIDDEN TESTS

## Ferdig!

Supert, da er du ferdig med denne øvingsoppaven.